# BamiBERT LoRA trích xuất và phân loại mức áp dụng

Huấn luyện bộ tìm cụm và nối đối số với trọng số dương 8, rồi huấn luyện bộ phân loại hồ sơ.

[Báo cáo tổng hợp](../reports/tong-hop-pipeline-smartnotice-v3.3-2026-11-23.md) · [Giao thức đánh giá](../../../docs/evaluation.md)

## 1. Cấu hình và dữ liệu

Nạp thông báo đã chuẩn bị và kiểm tra hash, nhóm, schema và các khoảng ký tự. Tham chiếu test v3.3 là chú giải lịch sử có máy hỗ trợ.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

repo = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tools/compare_notice_pipelines.py").is_file()),
    None,
)
if repo is None:
    raise RuntimeError("Mở notebook từ workspace SmartNotice.")
os.chdir(repo)
split = Path(os.environ.get("SMARTNOTICE_SPLIT_ROOT", "tmp/drive-run-20261123-fresh")).resolve()
base = Path(os.environ.get("SMARTNOTICE_BAMIBERT_DIR", "tmp/bamibert-checkpoint")).resolve()
destination = Path(os.environ.get(
    "SMARTNOTICE_NOTEBOOK_OUTPUT", "reports/generated/pipeline-notebooks-v33-20261123"
)).resolve()
baseline = Path(os.environ.get(
    "SMARTNOTICE_BASELINE", "reports/generated/bamibert/v3_3-2026-11-23-final-seed42"
)).resolve()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def run_tool(script, *arguments, **environment):
    env = {**os.environ, "PYTHONHASHSEED": "42", "TOKENIZERS_PARALLELISM": "false", **environment}
    result = subprocess.run(
        [sys.executable, "-u", str(repo / "tools" / script), *map(str, arguments)],
        cwd=repo, env=env, capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    if result.returncode:
        print(result.stdout[-4000:], result.stderr[-4000:])
        result.check_returncode()
    print(result.stdout.strip())

In [2]:
from smartnotice.modeling.data import labeled_links, load_development, sha256, span_key

development, provenance = load_development(split)
for part, docs in development.items():
    print(part, {
        "notices": len(docs),
        "groups": len({d.group_id for d in docs}),
        "typed_spans": sum(len({span_key(s) for s in d.annotation.spans}) for d in docs),
        "links": sum(len(set(labeled_links(d.annotation))) for d in docs),
        "events": sum(len(d.annotation.events) for d in docs),
    })
print("Phân bổ:", provenance["split_counts"], "gold_ready:", provenance["gold_ready"])

train {'notices': 403, 'groups': 281, 'typed_spans': 4664, 'links': 3847, 'events': 1516}
validation {'notices': 93, 'groups': 77, 'typed_spans': 1078, 'links': 873, 'events': 357}
Phân bổ: {'train': 403, 'test': 126, 'validation': 93, 'domain_test': 62} gold_ready: False


## 2. Token hóa và huấn luyện

Dùng các bước của [notebook thực nghiệm](../legacy/01_bamibert_development.ipynb).
Bộ trích xuất học trên train và chọn ngưỡng theo F1 liên kết validation. Bộ phân loại hồ sơ học bằng một checkpoint BamiBERT riêng.
Có thể đặt SMARTNOTICE_RESUME_EXTRACTION tới checkpoint đã xác minh để huấn luyện lại riêng bộ phân loại hồ sơ.

In [3]:
import gc
import hashlib
import json
import os
import platform
import shutil
from dataclasses import asdict
from importlib.metadata import version
from importlib.resources import files
from pathlib import Path

import torch
from smartnotice.modeling.applicability import train_applicability
from smartnotice.modeling.data import choose_width, load_development, sha256, tokenize
from smartnotice.modeling.experiment import (
    Settings,
    evaluate_extraction,
    load_extractor,
    train_extraction,
    write_summary,
)
from transformers import AutoTokenizer

MODEL = "Qualcomm-AI-Research/BamiBERT"
REVISION = "57bc1340debbe4e348ec549047a763caebe4a977"
MODEL_SHA256 = "c8c8abe5914f2543bd6a65019b685dacf25ad39da5f01b680b082876e1947627"
root = split
output = destination / "run-01"
output.mkdir(parents=True, exist_ok=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
settings = Settings(
    tuning="lora",
    suppress_same_type_overlap=False,
)
print(device, flush=True)
checkpoint = str(base)
checkpoint_path = Path(checkpoint)
weights_sha256 = sha256(checkpoint_path / "model.safetensors")
if weights_sha256 != MODEL_SHA256:
    raise ValueError("BamiBERT weights do not match the pinned release")
tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
assert tokenizer.is_fast
print(tokenizer.__class__.__name__)

cuda


XLMRobertaTokenizerFast


In [4]:
documents, provenance = load_development(root)
width = choose_width(documents["train"], tokenizer, settings.length, settings.overlap)
tokenized = {
    split: [tokenize(doc, tokenizer, settings.length, settings.overlap, width) for doc in items]
    for split, items in documents.items()
}
run = {
    "model": MODEL,
    "revision": REVISION,
    "settings": asdict(settings),
    "model_weights_sha256": weights_sha256,
    "tokenizer_sha256": sha256(checkpoint_path / "tokenizer.json"),
    "tokenizer_class": tokenizer.__class__.__name__,
    "data": provenance,
    "max_width_subwords": width,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "device": str(device),
    "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
    "versions": {name: version(name) for name in ("torch", "transformers", "scikit-learn")},
    "peft_version": version("peft") if settings.tuning == "lora" else None,
    "code_sha256": {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest()
        for p in files("smartnotice.modeling").iterdir()
        if p.name.endswith(".py")
    },
    "label_scope": "development only",
    "counts": {
        split: {
            "documents": len(items),
            "windows": sum(len(d.windows) for d in items),
            "groups": len({d.document.group_id for d in items}),
        }
        for split, items in tokenized.items()
    },
}
write_summary(output / "run.json", run)
print(run["counts"], "max width:", width)

{'train': {'documents': 403, 'windows': 1221, 'groups': 281}, 'validation': {'documents': 93, 'windows': 284, 'groups': 77}} max width: 60


In [5]:
from dataclasses import asdict

reuse = os.environ.get("SMARTNOTICE_FROZEN_EXTRACTION")
if reuse:
    prior = Path(reuse).resolve()
    previous = read_json(prior / "run.json")
    for field in ("data", "model_weights_sha256", "tokenizer_sha256", "max_width_subwords"):
        if previous[field] != run[field]:
            raise ValueError(f"Checkpoint metadata differs: {field}")
    if Settings(**previous["settings"]) != settings:
        raise ValueError("Checkpoint training settings differ")
    shutil.copyfile(prior / "extraction.pt", output / "extraction.pt")
    extraction = read_json(prior / "extraction_metrics.json")
    write_summary(output / "extraction_metrics.json", extraction)
    run["frozen_extraction"] = {
        "source": str(prior.relative_to(repo)) if prior.is_relative_to(repo) else prior.name,
        "sha256": sha256(output / "extraction.pt"),
        "settings": asdict(settings),
    }
    write_summary(output / "run.json", run)
    print("Nap trong so trich xuat da huan luyen, SHA256", run["frozen_extraction"]["sha256"])
    print("Nguong da chot tren validation:", extraction["thresholds"])
else:
    resume = os.environ.get("SMARTNOTICE_RESUME_EXTRACTION")
    if resume:
        prior = Path(resume)
        prior_run = json.loads((prior / "run.json").read_text(encoding="utf-8"))
        for key in ("model_weights_sha256", "tokenizer_sha256", "data", "max_width_subwords"):
            if prior_run[key] != run[key]:
                raise ValueError(f"Extraction run input differs: {key}")
        if Settings(**prior_run["settings"]) != settings:
            raise ValueError("Extraction run settings differ")
        saved = torch.load(prior / "extraction.pt", map_location="cpu")
        model = load_extractor(checkpoint, saved, tokenizer.pad_token_id, settings, device)
        extraction, predictions = evaluate_extraction(model, tokenized["validation"], settings)
        del model
        shutil.copyfile(prior / "extraction.pt", output / "extraction.pt")
        run["resumed_extraction"] = {
            "checkpoint_sha256": sha256(output / "extraction.pt"),
            "training_epoch_selected": saved["epoch"],
            "training_run_sha256": sha256(prior / "run.json"),
        }
        write_summary(output / "run.json", run)
        write_summary(output / "extraction_metrics.json", extraction)
    else:
        extraction, predictions = train_extraction(
            tokenized["train"],
            tokenized["validation"],
            checkpoint,
            tokenizer.pad_token_id,
            width,
            settings,
            output,
            device,
        )
    for key in ("span", "link_annotated_endpoints", "link_predicted_endpoints", "exact_event"):
        print(key, extraction[key].get("micro", extraction[key]))
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

{'epoch': 1, 'joint': False, 'loss': 0.45728674227783167}


{'epoch': 2, 'joint': True, 'loss': 1.0349557179190385, 'validation_link_f1': 0.008152173913043478}


{'epoch': 3, 'joint': True, 'loss': 0.9273827054050218, 'validation_link_f1': 0.007334963325183374}


{'epoch': 4, 'joint': True, 'loss': 0.8611118605559133, 'validation_link_f1': 0.006535947712418301}


span {'precision': 0.04109707561861914, 'recall': 0.44063079777365494, 'f1': 0.07518201962646408, 'matched': 475, 'predicted': 11558, 'expected': 1078}
link_annotated_endpoints {'precision': 0.5022900763358779, 'recall': 0.3768613974799542, 'f1': 0.4306282722513089, 'matched': 329, 'predicted': 655, 'expected': 873}
link_predicted_endpoints {'precision': 0.01001669449081803, 'recall': 0.006872852233676976, 'f1': 0.008152173913043478, 'matched': 6, 'predicted': 599, 'expected': 873}
exact_event {'precision': 0.0031631269769543608, 'recall': 0.0392156862745098, 'f1': 0.005854066485469371, 'matched': 14, 'predicted': 4426, 'expected': 357}


In [6]:
if reuse:
    for name in ("applicability.pt", "applicability_metrics.json"):
        shutil.copyfile(prior / name, output / name)
    run["frozen_applicability_sha256"] = sha256(output / "applicability.pt")
    write_summary(output / "run.json", run)
    print("Nap bo phan loai ap dung da huan luyen.")
else:
    applicability = train_applicability(
        documents["train"],
        documents["validation"],
        predictions,
        checkpoint,
        tokenizer,
        settings,
        output,
        device,
    )
    write_summary(
        output / "metrics.json",
        {
            "run": run,
            "extraction": extraction,
            "applicability": applicability,
        },
    )
    print("Annotated-event applicability Macro-F1:", applicability["annotated_events"]["macro_f1"])
    print(
        "Matched predicted-event applicability:",
        applicability["predicted_events"]["matched_action_applicability"]["macro_f1"],
    )

{'epoch': 1, 'loss': 0.9259776993375409, 'validation_macro_f1': 0.5345553481740057}


{'epoch': 2, 'loss': 0.7061702637235735, 'validation_macro_f1': 0.670700724459115}


Annotated-event applicability Macro-F1: 0.670700724459115
Matched predicted-event applicability: 0.4098385537605782


In [7]:
out = output
app_run = output

## 3. Suy luận test và chấm điểm

In [8]:
from smartnotice.modeling.experiment import write_summary
from smartnotice.modeling.metrics import prf


def show(scores):
    span = scores["span"]["micro"]
    link = scores["link"]["micro"]
    event = scores["event"]
    for caption, score in (("Cụm có kiểu", span), ("Quan hệ", link), ("Sự kiện", event)):
        print(
            f"{caption}: F1 {score['f1']:.2%} | P {score['precision']:.2%} | "
            f"R {score['recall']:.2%} | đúng/dự đoán/tham chiếu "
            f"{score['matched']}/{score['predicted']}/{score['expected']}"
        )
    print("Ngày tường minh:", scores["dates"]["explicit"])


def without_reference(docs):
    from smartnotice.modeling.data import Document
    from smartnotice.schemas import Annotation

    return [
        Document(d.notice, Annotation(
            doc_id=d.notice.doc_id, annotator_id="inference", spans=[], events=[]
        ), d.group_id)
        for d in docs
    ]


def boundary_diagnostics(docs, predictions):
    expected, predicted = {}, {}
    for d in docs:
        doc_id = d.notice.doc_id
        for target, spans in (
            (expected, map(span_key, d.annotation.spans)), (predicted, predictions[doc_id][0])
        ):
            for begin, end, kind in spans:
                target.setdefault((doc_id, begin, end), set()).add(kind)
    matched = expected.keys() & predicted.keys()
    return {
        "untyped": prf(set(expected), set(predicted)),
        "correct_type_sets": sum(expected[key] == predicted[key] for key in matched),
        "matched_boundaries": len(matched),
    }

In [9]:
import torch
from transformers import AutoTokenizer
from tools.compare_notice_pipelines import _bert_predictions

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(base, use_fast=True)

In [10]:
from tools.compare_notice_pipelines import _metrics, _test_docs

test = _test_docs(split)
print("Test:", len(test), "thông báo,", len({d.group_id for d in test}), "nhóm")

Test: 126 thông báo, 118 nhóm


In [11]:
weights = out / "extraction.pt"

predictions = _bert_predictions(weights, without_reference(test), base, tokenizer, device)
result = _metrics(test, predictions)
result["boundaries"] = boundary_diagnostics(test, predictions)
show(result)
print("Ranh giới và gán kiểu có điều kiện:", result["boundaries"])
write_summary(out / "test_scores.json", result)

Cụm có kiểu: F1 7.45% | P 4.08% | R 43.27% | đúng/dự đoán/tham chiếu 611/14982/1412
Quan hệ: F1 1.36% | P 1.69% | R 1.13% | đúng/dự đoán/tham chiếu 13/767/1149
Sự kiện: F1 0.31% | P 0.17% | R 1.92% | đúng/dự đoán/tham chiếu 9/5271/468
Ngày tường minh: {'expected_links': 188, 'matched_links': 0, 'correct_dates': 0, 'matched_date_accuracy': None, 'matched_inferred_flag_accuracy': None, 'end_to_end_date_accuracy': 0.0}
Ranh giới và gán kiểu có điều kiện: {'untyped': {'precision': 0.04191696702709918, 'recall': 0.4447592067988669, 'f1': 0.07661339514456508, 'matched': 628, 'predicted': 14982, 'expected': 1412}, 'correct_type_sets': 611, 'matched_boundaries': 628}


In [12]:
if os.environ.get("SMARTNOTICE_SKIP_APPLICABILITY") == "1":
    print("Applicability evaluation disabled for this run.")
else:
    from types import SimpleNamespace
    from tools.compare_notice_pipelines import _app

    app = _app(
        SimpleNamespace(base=base, baseline=app_run),
        development["train"], test, {"pipeline": predictions}, device, tokenizer,
    )["pipeline"]
    result["applicability"] = app
    print("F1 sự kiện và nhãn áp dụng:", f"{app['joint_exact_event_profile']['f1']:.2%}")
    write_summary(out / "test_scores.json", result)

F1 sự kiện và nhãn áp dụng: 0.31%
